In [2]:
here::i_am("rna/trajectories/infer_trajectory.R")
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/code



In [3]:
io$basedir

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome"

In [22]:
chip = fread('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/results/rna_atac/gene_regulatory_networks/metacells/trajectories/blood/TF2gene_after_virtual_chip.txt.gz')
io$outdir = file.path(io$basedir,"results/rna_atac/gene_regulatory_networks/metacells/trajectories/blood/in_silico_chip")
dir.create(io$outdir, showWarnings=F, recursive=T)


In [23]:
TFs = c('TAL1', 'CEBPB', 'GATA2', 'LMO2', 'FLI1', 'MEIS1', 'GFI1', 'GFI1B', 'RUNX1', 'SPI1', 'POU5F1', 'ELK4')
TFs = TFs[TFs %in% chip$tf]

In [39]:
for(i in TFs){
    chip_bed = chip[tf==i, c('peak', 'chip_score')] %>% 
        unique(by='peak') %>%
        setnames(c('peak', 'score')) %>%
        .[,peak:=str_replace(peak,":","-")] %>%
        .[,chr:=strsplit(peak,"-") %>% map_chr(1)] %>%
        .[,start:=strsplit(peak,"-") %>% map_chr(2)] %>%
        .[,end:=strsplit(peak,"-") %>% map_chr(3)] %>%
        .[,name:=i] %>%
        .[,c("chr","start","end","name","score")]
    tf = i 
    chip_bed = rbind(data.table("chr"=paste0('track type=bed name=', paste0(tf, '_Silico_chip'), ' description=', paste0(tf, '_Silico_chip'), ' visibility=1 windowingFunction=maximum color=', '31,120,180',"start","end","name","score"),
                     "start"=NA,"end"=NA,"name"=NA,"score"=NA), chip_bed)
        
    fwrite(chip_bed, sprintf("%s/%s.bed",io$outdir,i), sep="\t", quote=F, col.names = F)
}

In [40]:
 chip_bed = chip[tf%in% TFs, c('tf', 'peak', 'chip_score')] %>% 
        unique(by=c('tf', 'peak')) %>%
        setnames(c('tf', 'peak', 'score')) %>%
        .[,peak:=str_replace(peak,":","-")] %>%
        .[,chr:=strsplit(peak,"-") %>% map_chr(1)] %>%
        .[,start:=strsplit(peak,"-") %>% map_chr(2)] %>%
        .[,end:=strsplit(peak,"-") %>% map_chr(3)] %>%
        .[,c("chr","start","end", 'tf', "score")]

In [41]:
    fwrite(chip_bed[,-5], sprintf("%s/all_TFs.bed",io$outdir), sep="\t", quote=F, col.names = F)


# Create browser tracks

In [17]:
files = list.files(io$outdir, pattern='bed')
files = files[-grep('.gz', files)]

In [20]:
beds = lapply(files, function(i){
    tf = strsplit(i, '\\.') %>% map_chr(1)
    paste0('track type=BED name=', paste0(tf, '_Silico_chip'), ' description=', paste0(tf, '_Silico_chip'), ' visibility=2 windowingFunction=maximum color=', '31,120,180', ' bigDataUrl=', paste0('http://bioinformatics.stemcells.cam.ac.uk/Files_for_transfer/Open/bart/haem_chip/', i))
    }) 

writeLines(unlist(beds), sprintf("%s/UCSC.txt",io$outdir))

In [42]:
io$outdir

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/gastrulation_multiome/results/rna_atac/gene_regulatory_networks/metacells/trajectories/blood/in_silico_chip"

[1] "GSM1692840_HB_Cebpb.bw"      "GSM1692841_HB_Gata2.bw"     
 [3] "GSM1692842_HB_Lmo2.bw"       "GSM1692843_HB_Tal1.bw"      
 [5] "GSM1692844_HE_Cebpb.bw"      "GSM1692845_HE_Fli1.bw"      
 [7] "GSM1692846_HE_Lmo2.bw"       "GSM1692847_HE_Meis1.bw"     
 [9] "GSM1692848_HE_Tal1.bw"       "GSM1692849_HP_Cebpb.bw"     
[11] "GSM1692850_HP_Fli1.bw"       "GSM1692851_HP_Gata1.bw"     
[13] "GSM1692852_HP_Gata2.bw"      "GSM1692853_HP_Gfi1.bw"      
[15] "GSM1692854_HP_Gfi1b.bw"      "GSM1692855_HP_Lmo2.bw"      
[17] "GSM1692856_HP_Runx1.bw"      "GSM1692857_HP_Spi1.bw"      
[19] "GSM1692858_HP_Tal1.bw"       "GSM1692859_MAC_Cebpb.bw"    
[21] "GSM1692860_MAC_Fli1.bw"      "GSM1692861_MAC_Lmo2.bw"     
[23] "GSM1692862_MAC_Runx1.bw"     "GSM1692863_MAC_Spi1.bw"     
[25] "GSM1692864_MAC_Tal1.bw"      "GSM1692865_MES_Cebpb.bw"    
[27] "GSM1692866_MES_Elk4.bw"      "GSM1692867_MES_Pou5F1.bw"   
[29] "GSM1968747_FlkPlus_Tead4.bw"

In [84]:
colors[TF=='TAL1']


TF,color
<chr>,<chr>
TAL1,"222,000,000"


In [90]:
files = list.files(io$outdir, pattern='bed')
files = files[-grep('.gz', files)]

In [91]:
files

[1] "all_TFs.bed" "CEBPB.bed"   "ELK4.bed"    "FLI1.bed"    "GATA2.bed"  
 [6] "GFI1B.bed"   "LMO2.bed"    "MEIS1.bed"   "POU5F1.bed"  "RUNX1.bed"  
[11] "SPI1.bed"    "TAL1.bed"

In [101]:
bigwigs = list.files('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/haem_chip', pattern='bw')
    colors = data.table(TF=c('TAL1', 'GATA2', 'FLI1', 'GFI1B', 'MEIS1','RUNX1','SPI1'), 
                        color=c('255,0,0', '0,0,255', '0,255,0','255,255,0','0,255,255','255,0,255', '100,100,200'))

for(i in c('TAL1', 'GATA2', 'FLI1', 'GFI1B', 'MEIS1','RUNX1','SPI1')){
    bigwigs_tf = bigwigs[grep(str_to_title(i), bigwigs)]
    bws = unlist(lapply(bigwigs_tf, function(tf){
        name = paste0(strsplit(tf, '_') %>% map_chr(2), '_', strsplit(tf, '_') %>% map_chr(3))
        colors_tf = colors[TF==i, color]
                paste0('track type=bigWig name=', name, 
                   ' description=', name, 
                   ' visibility=2 windowingFunction=maximum color=', colors_tf, 
                   ' bigDataUrl=', paste0('http://bioinformatics.stemcells.cam.ac.uk/Files_for_transfer/Open/bart/haem_chip/', tf))
        }))
    writeLines(unlist(bws), sprintf("%s/UCSC_%s.txt",io$outdir, i))
    }

In [99]:
bws

[1] "track type=bigWig name=HB_Gata2.bw description=HB_Gata2.bw visibility=2 windowingFunction=maximum color= bigDataUrl=http://bioinformatics.stemcells.cam.ac.uk/Files_for_transfer/Open/bart/haem_chip/GSM1692841_HB_Gata2.bw"
[2] "track type=bigWig name=HP_Gata2.bw description=HP_Gata2.bw visibility=2 windowingFunction=maximum color= bigDataUrl=http://bioinformatics.stemcells.cam.ac.uk/Files_for_transfer/Open/bart/haem_chip/GSM1692852_HP_Gata2.bw"

In [100]:
writeLines(unlist(bws), sprintf("%s/UCSC_%s.txt",io$outdir, i))

In [65]:
bigwigs = list.files('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/haem_chip', pattern='bw')
#lapply('GATA1', function(i){
    bigwigs_tf = bigwigs[grep(str_to_title(i), bigwigs)]
   # bws = unlist(lapply(bigwigs_tf, function(tf){
        paste0('track type=bigWig name=', paste0(tf, '_Silico_chip'), 
               ' description=', paste0(tf, '_Silico_chip'), 
               ' visibility=2 windowingFunction=maximum color=', '31,120,180', 
               ' bigDataUrl=', paste0('http://bioinformatics.stemcells.cam.ac.uk/Files_for_transfer/Open/bart/haem_chip/', tf))
    # }))
    # })

ERROR: Error in parse(text = x, srcfile = src): <text>:11:0: unexpected end of input
9:     # }))
10:     # })
   ^


In [61]:
i ='LMO2'
bigwigs_tf = bigwigs[grep(str_to_title(i), bigwigs)]
#bigwigs_tf
bigwigs_tf

[1] "GSM1692842_HB_Lmo2.bw"  "GSM1692846_HE_Lmo2.bw"  "GSM1692855_HP_Lmo2.bw" 
[4] "GSM1692861_MAC_Lmo2.bw"

In [47]:
list.files('/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/haem_chip')

[1] "GSE69099_RAW.tar"                "GSM1692840_HB_Cebpb.bed.gz"     
 [3] "GSM1692840_HB_Cebpb.bw"          "GSM1692841_HB_Gata2.bed.gz"     
 [5] "GSM1692841_HB_Gata2.bw"          "GSM1692842_HB_Lmo2.bed.gz"      
 [7] "GSM1692842_HB_Lmo2.bw"           "GSM1692843_HB_Tal1.bed.gz"      
 [9] "GSM1692843_HB_Tal1.bw"           "GSM1692844_HE_Cebpb.bed.gz"     
[11] "GSM1692844_HE_Cebpb.bw"          "GSM1692845_HE_Fli1.bed.gz"      
[13] "GSM1692845_HE_Fli1.bw"           "GSM1692846_HE_Lmo2.bed.gz"      
[15] "GSM1692846_HE_Lmo2.bw"           "GSM1692847_HE_Meis1.bed.gz"     
[17] "GSM1692847_HE_Meis1.bw"          "GSM1692848_HE_Tal1.bed.gz"      
[19] "GSM1692848_HE_Tal1.bw"           "GSM1692849_HP_Cebpb.bed.gz"     
[21] "GSM1692849_HP_Cebpb.bw"          "GSM1692850_HP_Fli1.bed.gz"      
[23] "GSM1692850_HP_Fli1.bw"           "GSM1692851_HP_Gata1.bed.gz"     
[25] "GSM1692851_HP_Gata1.bw"          "GSM1692852_HP_Gata2.bed.gz"     
[27] "GSM1692852_HP_Gata2.bw"          "GSM1692853_HP_Gfi1.bed.gz"      
[29] "GSM1692853_HP_Gfi1.bw"           "GSM1692854_HP_Gfi1b.bed.gz"     
[31] "GSM1692854_HP_Gfi1b.bw"          "GSM1692855_HP_Lmo2.bed.gz"      
[33] "GSM1692855_HP_Lmo2.bw"           "GSM1692856_HP_Runx1.bed.gz"     
[35] "GSM1692856_HP_Runx1.bw"          "GSM1692857_HP_Spi1.bed.gz"      
[37] "GSM1692857_HP_Spi1.bw"           "GSM1692858_HP_Tal1.bed.gz"      
[39] "GSM1692858_HP_Tal1.bw"           "GSM1692859_MAC_Cebpb.bed.gz"    
[41] "GSM1692859_MAC_Cebpb.bw"         "GSM1692860_MAC_Fli1.bed.gz"     
[43] "GSM1692860_MAC_Fli1.bw"          "GSM1692861_MAC_Lmo2.bed.gz"     
[45] "GSM1692861_MAC_Lmo2.bw"          "GSM1692862_MAC_Runx1.bed.gz"    
[47] "GSM1692862_MAC_Runx1.bw"         "GSM1692863_MAC_Spi1.bed.gz"     
[49] "GSM1692863_MAC_Spi1.bw"          "GSM1692864_MAC_Tal1.bed.gz"     
[51] "GSM1692864_MAC_Tal1.bw"          "GSM1692865_MES_Cebpb.bed.gz"    
[53] "GSM1692865_MES_Cebpb.bw"         "GSM1692866_MES_Elk4.bed.gz"     
[55] "GSM1692866_MES_Elk4.bw"          "GSM1692867_MES_Pou5F1.bed.gz"   
[57] "GSM1692867_MES_Pou5F1.bw"        "GSM1968747_FlkPlus_Tead4.bed.gz"
[59] "GSM1968747_FlkPlus_Tead4.bw"